In [1]:
# Cell 1: Load libraries, data and trained models
# SHAP needs the trained models to explain their predictions

import numpy as np # for numerical operations and loading .npy files(used for arrays and matric calculations)
import pandas as pd # for CSV files and handling table(like excel spreadsheets)
import shap # explain WHY a ML model makes a prediction by measuring feature importance
import matplotlib # make graphs and plots
matplotlib.use('Agg')  # prevents display errors on Mac
import matplotlib.pyplot as plt # main for plotting tool from matplotlib, use for creating SHAP lots
import json # for reading or saving JSON formatted files
import time # use for measure execution time(start timer and check how long code runs)

# path where all processed files and models are saved
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"

# LOAD TEST DATASET
# SHAP explains predictions on TEST data (unseen data)
# X_train not needed here — models already trained
# Shape example: (10000 samples,  77 features) ; 10000 samples with 77 features
X_test  = np.load(save_path + "X_test.npy") # np.load()  for numpy binary files

# pd.read(): reads CSV file into a pandas DataFrame
# .squeeze(): convert one-column table -> series: y_test contain true labbels: normal, attach, normal, attach...
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze() 


# LOAD CLASS NAMES AND FEAUTRE NAMES
# label_classes.csv stores attack categories: ['Normal', 'Dos', 'Spoofing']
# tolist(): converts pandas object -> python list
le_classes    = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist() # squeeze(): to remove unnecessary table dimension
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist() # feature_names.csv stores names of all features

# LOAD TRAINED MODELS
# joblib.load() reads to saved / load sklearn models (.joblib files)
import joblib
rf_model = joblib.load(save_path + "rf_model.joblib") # load trained RF model
print("RF model loaded!")

# LOAD DL models
# keras.models.load_model() reads saved TensorFlow models (.keras files)
from tensorflow import keras
cnn_model = keras.models.load_model(save_path + "cnn_model.keras") # load trained 1D-CNN model
ae_model  = keras.models.load_model(save_path + "ae_model.keras")
print("CNN model loaded!")
print("AE model loaded!")

# LOAD ANTOENCODER THRESHOLD 
# error > threshold -> attack; error <= threshold -> normal
# load threshold saved during autoencoder training
# needed to convert AE reconstruction error → binary prediction
ae_threshold = np.load(save_path + "ae_threshold.npy")[0]
print(f"AE threshold loaded: {ae_threshold:.6f}")

print(f"\nX_test: {X_test.shape}")
print(f"Classes: {le_classes}")
print(f"Features: {len(feature_names)}")
print("\nAll loaded!")

RF model loaded!


2026-05-11 10:50:12.867765: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CNN model loaded!
AE model loaded!
AE threshold loaded: 2.250115

X_test: (9931, 37)
Classes: ['DoS attack', 'Replay', 'benign']
Features: 37

All loaded!


In [15]:
# Cell 2: Apply SHAP to Random Forest
#
# KEY CONCEPT: SHAP (SHapley Additive exPlanations)
# SHAP assigns each feature an importance value for each prediction
# Based on game theory — how much does each feature contribute?
#
# TreeExplainer = fast SHAP method designed for tree-based models
# Why TreeExplainer for RF? Because RF is made of decision trees
# Much faster than KernelExplainer for tree models
#
# shap_values shape: (samples, features, classes)
# We use mean absolute value across samples for global importance

print("Running SHAP on Random Forest...")

# use small sample for speed
# SHAP is slow on large datasets — 500 samples is enough for global view
sample_size = 500
np.random.seed(42)  # fixed seed for reproducibility
sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test[sample_idx]

start = time.time()

# TreeExplainer: optimized for decision tree models (RF, XGBoost, etc)
explainer_rf = shap.TreeExplainer(rf_model)

# shap_values: importance of each feature for each prediction
# shape: (samples, features, classes) for multiclass
shap_values_rf = explainer_rf.shap_values(X_sample)

elapsed_rf = round(time.time() - start, 2)
print(f"SHAP RF complete! Time: {elapsed_rf}s")

# calculate global importance
# shap_values_rf shape: (500, 37, 3)
# axis=0 = average across samples → (37, 3)
# axis=1 = average across classes → (37,)
shap_rf_global = np.abs(shap_values_rf).mean(axis=0).mean(axis=1)

# get top 10 features sorted by importance
top_idx_rf = np.argsort(shap_rf_global)[::-1][:10].tolist()

print(f"\nTop 10 Features — RF SHAP:")
print(f"{'Rank':<6} {'Feature':<35} {'SHAP Value':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_rf):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {shap_rf_global[idx]:.6f}")

# plot global feature importance
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_rf[::-1]],  # feature names
    shap_rf_global[top_idx_rf[::-1]],               # importance values
    color='steelblue'
)
plt.xlabel('Mean |SHAP Value|')
plt.title('SHAP Global Feature Importance — Random Forest (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "shap_rf_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("RF SHAP plot saved!")

Running SHAP on Random Forest...
SHAP RF complete! Time: 13.34s

Top 10 Features — RF SHAP:
Rank   Feature                             SHAP Value     
--------------------------------------------------------
  1    timestamp_c                         0.250590
  2    wlan.fc.type                        0.045735
  3    frame.number                        0.037830
  4    wlan.duration                       0.034963
  5    wlan.seq                            0.014748
  6    wlan.ta                             0.012282
  7    wlan.sa                             0.011887
  8    wlan.fc.subtype                     0.011287
  9    frame.len                           0.009176
  10   udp.dstport                         0.007811
RF SHAP plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1890/646829779.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Cell 3: Apply SHAP to 1D-CNN
#
# KernelExplainer = model-agnostic SHAP method
# Works on ANY model — treats model as black box
# Why KernelExplainer for CNN? 
# CNN is not a tree model so TreeExplainer won't work
# KernelExplainer works by perturbing input features
#
# IMPORTANT: CNN needs 3D input (samples, features, 1)
# But KernelExplainer sends 2D input (samples, features)
# We need a WRAPPER FUNCTION to reshape input before prediction

print("Running SHAP on 1D-CNN...")
print("Using 100 samples (KernelExplainer is slower than TreeExplainer)...")

# smaller sample for CNN — KernelExplainer is much slower
sample_size_cnn = 100
X_sample_cnn = X_test[sample_idx[:sample_size_cnn]]

# background data: small sample used as baseline reference
# SHAP compares each prediction against this baseline
# 50 samples is enough for stable estimates
background = X_test[:50]

# wrapper function: reshapes 2D → 3D for CNN
# reason: KernelExplainer passes 2D data but CNN needs 3D
def cnn_predict(X):
    # X shape coming in: (samples, 37) ← 2D
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # → (samples, 37, 1) ← 3D
    # predict() returns probabilities for each class
    return cnn_model.predict(X_3d, verbose=0)

start = time.time()

# KernelExplainer: works with any model via wrapper function
explainer_cnn = shap.KernelExplainer(cnn_predict, background)

# shap_values: list of arrays, one per class
# nsamples='auto' = automatically choose number of perturbations
shap_values_cnn = explainer_cnn.shap_values(X_sample_cnn, nsamples='auto')

elapsed_cnn = round(time.time() - start, 2)
print(f"SHAP CNN complete! Time: {elapsed_cnn}s")

# calculate global importance across all classes
shap_cnn_global = np.mean([np.abs(sv).mean(axis=0) 
                            for sv in shap_values_cnn], axis=0)

# get top 10 features
top_idx_cnn = np.argsort(shap_cnn_global)[::-1][:10]

print(f"\nTop 10 Features — CNN SHAP:")
print(f"{'Rank':<6} {'Feature':<35} {'SHAP Value':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_cnn):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {shap_cnn_global[idx]:.6f}")

# plot
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_cnn[::-1]],
    shap_cnn_global[top_idx_cnn[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |SHAP Value|')
plt.title('SHAP Global Feature Importance — 1D-CNN (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "shap_cnn_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("CNN SHAP plot saved!")

Running SHAP on 1D-CNN...
Using 100 samples (KernelExplainer is slower than TreeExplainer)...


  0%|          | 0/100 [00:00<?, ?it/s]

SHAP CNN complete! Time: 1063.95s

Top 10 Features — CNN SHAP:
Rank   Feature                             SHAP Value     
--------------------------------------------------------
  1    frame.number                        0.015723
  2    timestamp_c                         0.013646
  3    frame.len                           0.012849
CNN SHAP plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1890/393366070.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Cell 4: Apply SHAP to Autoencoder
#
# Autoencoder is different from RF and CNN:
# RF and CNN predict CLASS labels (DoS, Replay, benign)
# Autoencoder predicts RECONSTRUCTION ERROR (MSE)
#
# For SHAP on Autoencoder:
# We explain the RECONSTRUCTION ERROR not class labels
# Higher SHAP value = feature contributes more to reconstruction error
# = feature is more important for anomaly detection

print("Running SHAP on Autoencoder...")
print("Using 50 samples (smallest — AE SHAP is slowest)...")

sample_size_ae = 50
X_sample_ae = X_test[sample_idx[:sample_size_ae]]

# background data for AE
background_ae = X_test[:30]

# wrapper function for Autoencoder
# returns MSE (reconstruction error) instead of class probabilities
# reason: AE doesn't predict classes — it predicts reconstruction quality
def ae_predict(X):
    # reconstruct input through encoder → decoder
    X_reconstructed = ae_model.predict(X, verbose=0)
    # calculate MSE per sample
    # np.mean(..., axis=1) = average error across all 37 features
    mse = np.mean(np.power(X - X_reconstructed, 2), axis=1)
    # reshape to 2D for SHAP compatibility
    return mse.reshape(-1, 1)

start = time.time()

explainer_ae = shap.KernelExplainer(ae_predict, background_ae)
shap_values_ae = explainer_ae.shap_values(X_sample_ae, nsamples='auto')

elapsed_ae = round(time.time() - start, 2)
print(f"SHAP AE complete! Time: {elapsed_ae}s")

# calculate global importance
shap_ae_global = np.abs(shap_values_ae[0]).mean(axis=0)

# get top 10 features
top_idx_ae = np.argsort(shap_ae_global)[::-1][:10]

print(f"\nTop 10 Features — AE SHAP:")
print(f"{'Rank':<6} {'Feature':<35} {'SHAP Value':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_ae):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {shap_ae_global[idx]:.6f}")

# plot
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_ae[::-1]],
    shap_ae_global[top_idx_ae[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |SHAP Value|')
plt.title('SHAP Global Feature Importance — Autoencoder (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "shap_ae_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("AE SHAP plot saved! ✅")

Running SHAP on Autoencoder...
Using 50 samples (smallest — AE SHAP is slowest)...


  0%|          | 0/50 [00:00<?, ?it/s]

SHAP AE complete! Time: 219.53s

Top 10 Features — AE SHAP:
Rank   Feature                             SHAP Value     
--------------------------------------------------------
  1    timestamp_c                         0.005374
AE SHAP plot saved! ✅


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1890/3109949215.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Cell 5: Save all SHAP results to JSON
# JSON format allows easy comparison across datasets later

# save RF SHAP results
shap_rf_results = {
    "model": "RandomForest",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "SHAP",
    "sample_size": sample_size,
    "time_seconds": elapsed_rf,
    "top_10_features": {
        feature_names[int(i)]: round(float(shap_rf_global[int(i)]), 6)
        for i in top_idx_rf
    }
}
with open(save_path + "shap_rf_results.json", "w") as f:
    json.dump(shap_rf_results, f, indent=2)
print("RF SHAP JSON saved!")

# save CNN SHAP results
shap_cnn_results = {
    "model": "1D-CNN",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "SHAP",
    "sample_size": sample_size_cnn,
    "time_seconds": elapsed_cnn,
    "top_10_features": {
        feature_names[i]: round(float(shap_cnn_global[i]), 6)
        for i in top_idx_cnn
    }
}
with open(save_path + "shap_cnn_results.json", "w") as f:
    json.dump(shap_cnn_results, f, indent=2)
print("CNN SHAP JSON saved!")

# save AE SHAP results
shap_ae_results = {
    "model": "Autoencoder",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "SHAP",
    "sample_size": sample_size_ae,
    "time_seconds": elapsed_ae,
    "top_10_features": {
        feature_names[i]: round(float(shap_ae_global[i]), 6)
        for i in top_idx_ae
    }
}
with open(save_path + "shap_ae_results.json", "w") as f:
    json.dump(shap_ae_results, f, indent=2)
print("AE SHAP JSON saved!")

print("\nAll SHAP results saved!")
print(f"\nSummary:")
print(f"  RF  SHAP #1: {feature_names[top_idx_rf[0]]}")
print(f"  CNN SHAP #1: {feature_names[top_idx_cnn[0]]}")
print(f"  AE  SHAP #1: {feature_names[top_idx_ae[0]]}")

RF SHAP JSON saved!
CNN SHAP JSON saved!
AE SHAP JSON saved!

All SHAP results saved!

Summary:
  RF  SHAP #1: timestamp_c
  CNN SHAP #1: frame.number
  AE  SHAP #1: timestamp_c


In [18]:
# Fix: convert each index to int
top_idx_rf = [int(i) for i in np.argsort(shap_rf_global)[::-1][:10]]

print(f"\nTop 10 Features — RF SHAP:")
print(f"{'Rank':<6} {'Feature':<35} {'SHAP Value':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_rf):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {shap_rf_global[idx]:.6f}")

# plot
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_rf[::-1]],
    [shap_rf_global[i] for i in top_idx_rf[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |SHAP Value|')
plt.title('SHAP Global Feature Importance — Random Forest (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "shap_rf_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("RF SHAP plot saved!")


Top 10 Features — RF SHAP:
Rank   Feature                             SHAP Value     
--------------------------------------------------------
  1    timestamp_c                         0.250590
  2    wlan.fc.type                        0.045735
  3    frame.number                        0.037830
  4    wlan.duration                       0.034963
  5    wlan.seq                            0.014748
  6    wlan.ta                             0.012282
  7    wlan.sa                             0.011887
  8    wlan.fc.subtype                     0.011287
  9    frame.len                           0.009176
  10   udp.dstport                         0.007811
RF SHAP plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1890/2032390008.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Debug: check shape of shap_rf_global
print(f"shap_rf_global type: {type(shap_rf_global)}")
print(f"shap_rf_global shape: {np.array(shap_rf_global).shape}")
print(f"shap_values_rf type: {type(shap_values_rf)}")
if isinstance(shap_values_rf, list):
    print(f"Number of classes: {len(shap_values_rf)}")
    print(f"Each class shape: {np.array(shap_values_rf[0]).shape}")
else:
    print(f"shap_values_rf shape: {np.array(shap_values_rf).shape}")

shap_rf_global type: <class 'numpy.ndarray'>
shap_rf_global shape: (37, 3)
shap_values_rf type: <class 'numpy.ndarray'>
shap_values_rf shape: (500, 37, 3)


In [17]:
# Fix: average across classes (axis=1) to get one score per feature
# shap_values_rf shape: (500, 37, 3)
# Step 1: np.abs() = take absolute values
# Step 2: .mean(axis=0) = average across 500 samples → (37, 3)
# Step 3: .mean(axis=1) = average across 3 classes → (37,)
shap_rf_global = np.abs(shap_values_rf).mean(axis=0).mean(axis=1)

print(f"shap_rf_global shape: {shap_rf_global.shape}")  # should be (37,)

# now get top 10 indices as plain integers
top_idx_rf = [int(i) for i in np.argsort(shap_rf_global)[::-1][:10]]

print(f"\nTop 10 Features — RF SHAP:")
print(f"{'Rank':<6} {'Feature':<35} {'SHAP Value':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_rf):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {shap_rf_global[idx]:.6f}")

# plot
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_rf[::-1]],
    [shap_rf_global[i] for i in top_idx_rf[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |SHAP Value|')
plt.title('SHAP Global Feature Importance — Random Forest (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "shap_rf_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("RF SHAP plot saved!")

shap_rf_global shape: (37,)

Top 10 Features — RF SHAP:
Rank   Feature                             SHAP Value     
--------------------------------------------------------
  1    timestamp_c                         0.250590
  2    wlan.fc.type                        0.045735
  3    frame.number                        0.037830
  4    wlan.duration                       0.034963
  5    wlan.seq                            0.014748
  6    wlan.ta                             0.012282
  7    wlan.sa                             0.011887
  8    wlan.fc.subtype                     0.011287
  9    frame.len                           0.009176
  10   udp.dstport                         0.007811
RF SHAP plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1890/2491188628.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
